# 📋 Plan-Execute + HIL với Deep Agents SDK

Notebook này demo cơ chế **Plan-Execute** của Deep Agents SDK:

1. **Plan** — Agent hiểu yêu cầu, tự động lên danh sách task (`write_todos`)
2. **Review** — User xem plan, có thể đồng ý hoặc sửa đổi
3. **Execute** — Agent tự động chạy từng task
4. **HIL** — Trong lúc execute, HIL cho các quyết định nhạy cảm

## Kiến trúc

```
User: "Phân tích database KH, gửi email chào mừng, tạo báo cáo..."
       │
       ▼
┌──────────────────────────────────────────────┐
│  Agent (create_deep_agent)                   │
│  • Tự động gọi write_todos để lập plan       │
│  • Trình plan cho user duyệt                 │
│  • Sau khi approve, execute từng task        │
│  • HIL interrupt cho operations nhạy cảm     │
└──────────────────────────────────────────────┘
       │
       ├── ✅ Task 1: Search database (safe)
       ├── ⛔ Task 2: Send email → chờ HIL
       └── ⛔ Task 3: Generate report (safe)


## 1. Cài Đặt Dependencies

In [1]:
# import subprocess
# import sys

# deps = [
#     "deepagents>=0.6.0",
#     "langchain-openai>=0.3.0",
#     "langgraph>=0.4.0",
#     "langchain-core>=0.3.0",
# ]

# for dep in deps:
#     print(f"Installing {dep}...")
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])
# print("✅ All dependencies installed!")

## 2. Import & Khởi Tạo Model

Dùng `ChatOpenAI` với model tự host (OpenAI-compatible).  
Sửa `MODEL_STRING` hoặc `base_url` cho phù hợp với môi trường của bạn.

In [2]:
import os
import json
from datetime import datetime

from langchain_core.tools import tool
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from deepagents import create_deep_agent
from dotenv import load_dotenv
load_dotenv()  


# ── Model config ──────────────────────────────────────────────
# Sửa lại theo môi trường của bạn
from langchain_openai import ChatOpenAI

v_glm46 = ChatOpenAI(
    model="v_glm46",
    base_url="https://genai.vnpay.vn/aigateway/llm_glm46/v1",
    temperature=0.7,
    max_tokens=100000,
)

MODEL = v_glm46  # ← Đổi model ở đây nếu cần

print(f"✅ Model configured: {MODEL.model}")

✅ Model configured: v_glm46


In [3]:
# Debug: check event structure
# (Đã fix xong — event['data']['messages'] thay vì event['messages'])

## 3. Định Nghĩa Tools

Các tools mô phỏng nghiệp vụ xử lý yêu cầu khách hàng:
- `search_customer` / `search_product` — **Safe**, không cần HIL
- `send_email_campaign` — **Dangerous**, cần human duyệt nội dung
- `generate_report` — **Safe**, không cần HIL

In [4]:
# ── Mock data ──
CUSTOMERS = {
    "C001": {"name": "Nguyen Van A", "email": "a@example.com", "plan": "Premium",  "spent_2026": 25000000},
    "C002": {"name": "Tran Thi B", "email": "b@example.com", "plan": "Basic",    "spent_2026": 3500000},
    "C003": {"name": "Le Van C",   "email": "c@example.com", "plan": "Enterprise","spent_2026": 120000000},
}
PRODUCTS = {
    "P001": {"name": "Cloud Storage Basic",   "price": 500000,  "stock": 100},
    "P002": {"name": "Cloud Storage Pro",     "price": 2000000, "stock": 50},
    "P003": {"name": "AI API Access",         "price": 5000000, "stock": 20},
}


# ── Safe tools (không cần HIL) ──
@tool
def search_customer(customer_id: str) -> str:
    """Tra cứu thông tin khách hàng theo ID."""
    result = CUSTOMERS.get(customer_id.upper())
    return json.dumps(result, indent=2, ensure_ascii=False) if result \
           else json.dumps({"error": f"Customer {customer_id} not found"})


@tool
def search_product(product_id: str) -> str:
    """Tra cứu thông tin sản phẩm theo ID."""
    result = PRODUCTS.get(product_id.upper())
    return json.dumps(result, indent=2, ensure_ascii=False) if result \
           else json.dumps({"error": f"Product {product_id} not found"})


@tool
def generate_report(report_type: str, data_summary: str) -> str:
    """Tạo báo cáo tổng kết."""
    return f"📄 Report '{report_type}' generated. Summary: {data_summary[:200]}"


# ── Dangerous tool (cần HIL) ──
@tool
def send_email_campaign(to: str, subject: str, body: str) -> str:
    """Gửi email chiến dịch cho khách hàng. Gọi tool này khi cần gửi email. Hệ thống sẽ tự động chờ người dùng duyệt."""
    print(f"  📧 Sending campaign email → {to}")
    return f"✅ Campaign email sent to {to}: '{subject}'"


tools = [search_customer, search_product, generate_report, send_email_campaign]
print(f"✅ {len(tools)} tools defined: {[t.name for t in tools]}")
print("   - Safe:     search_customer, search_product, generate_report")
print("   - Dangerous: send_email_campaign (interrupt_on)")

✅ 4 tools defined: ['search_customer', 'search_product', 'generate_report', 'send_email_campaign']
   - Safe:     search_customer, search_product, generate_report
   - Dangerous: send_email_campaign (interrupt_on)


## 4. Tạo Deep Agent Plan-Execute + HIL

**Cơ chế Plan-Execute của Deep Agents SDK hoạt động thế nào?**

Deep Agents có built-in tool **`write_todos`** — agent tự động gọi nó để:
1. **Phân tích** yêu cầu, bẻ thành các task nhỏ
2. **Lên plan** với mô tả từng bước
3. **Execute** từng task theo thứ tự

Kết hợp với `interrupt_on` để HIL các tool nguy hiểm trong lúc execute.

> ⚠️ Plan-Execute là behavior có sẵn của Deep Agents.  
> Bạn chỉ cần system prompt hướng dẫn agent "plan trước, execute sau" + bật HIL.

In [16]:
checkpointer = MemorySaver()

agent = create_deep_agent(
    model=MODEL,
    tools=tools,
    checkpointer=checkpointer,
    interrupt_on={
        "send_email_campaign": {
            "allowed_decisions": ["approve", "edit", "reject"]
        },
    },
    system_prompt="""You are an intelligent customer operations agent.

## CRITICAL RULES — Follow exactly:

### 1. PLAN FIRST
Use `write_todos` to break requests into clear, actionable tasks.
Include ALL steps: search, analysis, email, reporting.

### 2. EXECUTE — CALL THE TOOL, DO NOT DESCRIBE IT
When a task says "send email" or "gửi email", you MUST:
- Call `send_email_campaign(to, subject, body)` directly
- Do NOT write the email content in your response and ask the user
- Do NOT ask "please confirm" or "please review"
- The system will automatically pause for human approval — you just call the tool

Example of what you MUST do:
  ✅ send_email_campaign(to="a@example.com", subject="Offer", body="...")

Example of what you MUST NOT do:
  ❌ "Here is the email content... Please confirm before I send."
  ❌ "I have drafted the email. Shall I send it?"

### 3. REPORT RESULTS
After each tool call, report the result clearly.
At the end, summarize everything you did.

Current date: """ + datetime.now().strftime("%Y-%m-%d"),
)

print("✅ Agent recreated!")
print("   → interrupt_on allows: approve, edit, reject")
print("   → Free-text dùng cơ chế user message riêng")

✅ Agent recreated!
   → interrupt_on allows: approve, edit, reject
   → Free-text dùng cơ chế user message riêng


In [6]:
# agent.invoke(
#     {
#         "messages": [
#             ('user', 
#              "Hãy lập kế hoạch và thực hiện chiến dịch email marketing cho khách hàng C001, C002, C003, giới thiệu sản phẩm P002 và P003. Tổng hợp kết quả cuối cùng."
#             )
#         ]
#     }
# )

## 5. Hàm HIL tương tác — Duyệt Plan & Tool Calls

Hàm `run_plan_execute()` sẽ:
1. Gọi agent với input của user
2. Agent tự động plan (gọi `write_todos`) và execute
3. Khi gặp tool dangerous → interrupt → user duyệt
4. Sau khi resume → agent tiếp tục execute các task còn lại

In [28]:
def get_human_decision(action, review_cfg):
    """Hỏi user quyết định.
    Trả về (decision_dict, free_text_or_None).
    - approve/edit/reject: decision_dict, None
    - free-text (f): reject_tool_decision + user_text → agent tự retry
    """
    name = action["name"]
    args = action["args"]
    allowed = review_cfg["allowed_decisions"]

    print(f"  Options: {', '.join(allowed)}")

    while True:
        choice = input("  Decision (a=approve, e=edit, r=reject, f=free-text): ").strip().lower()

        if choice in ("approve", "a"):
            return {"type": "approve"}, None
        elif choice in ("edit", "e"):
            if "edit" not in allowed:
                print("  ❌ Edit not allowed. Choose another.")
                continue
            new_args = {}
            for k, v in args.items():
                new_v = input(f"  New {k} [{v}]: ").strip()
                new_args[k] = new_v if new_v else v
            return {"type": "edit", "edited_action": {"name": name, "args": new_args}}, None
        elif choice in ("reject", "r"):
            if "reject" not in allowed:
                print("  ❌ Reject not allowed. Choose another.")
                continue
            reason = input("  Reason (optional): ").strip()
            if not reason:
                reason = f"User rejected {name}. Do not retry."
            return {"type": "reject", "message": reason}, None
        elif choice in ("free-text", "f"):
            text = input("  Your message to agent: ").strip()
            # Reject + text riêng → tool không chạy, agent đọc text và tự retry
            return {"type": "reject", "message": text}, text
        else:
            print(f"  ❌ Invalid. Must be: approve/a, edit/e, reject/r, free-text/f")


def stream_agent(user_input: str, thread_id: str):
    """Stream agent execution — hiển thị plan, tool calls, và kết quả step-by-step.
    
    Dùng stream_mode='values' để lấy toàn bộ state sau mỗi bước.
    """
    config = {"configurable": {"thread_id": thread_id}}

    print(f"👤 User: {user_input}")
    print("═" * 70)

    seen_tool_calls = set()

    for event in agent.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config,
        stream_mode="values",
        version="v2",
    ):
        # Messages nằm trong event['data']['messages'], không phải event['messages']
        msgs = event.get("data", {}).get("messages", [])
        if not msgs:
            continue
        last = msgs[-1]

        # Tool call — highlight todos vs regular tools
        if hasattr(last, "tool_calls") and last.tool_calls:
            for tc in last.tool_calls:
                # Dedup bằng id
                tc_id = tc.get("id", "")
                if tc_id in seen_tool_calls:
                    continue
                seen_tool_calls.add(tc_id)

                if tc["name"] == "write_todos":
                    todos = tc["args"].get("todos", [])
                    print(f"\n📋 PLAN — {len(todos)} task(s):")
                    for i, t in enumerate(todos, 1):
                        # write_todos có thể dùng content, title, hoặc description
                        label = t.get('title') or t.get('description') or t.get('content', '?')
                        print(f"   {i}. {label}")
                else:
                    args_str = json.dumps(tc["args"], ensure_ascii=False)
                    print(f"\n🔧 {tc['name']}({args_str})")

        # Tool result
        if hasattr(last, "type") and last.type == "tool":
            print(f"   ✅ Result: {last.content[:200]}")

        # Agent trả lời text (chỉ khi không phải republish)
        if isinstance(last, AIMessage) and last.content and not last.tool_calls:
            print(f"\n🤖 Agent: {last.content[:400]}")

    print("\n" + "═" * 70)


def run_plan_execute(user_input: str, thread_id: str):
    """Full Plan-Execute + HIL loop."""
    config = {"configurable": {"thread_id": thread_id}}

    print(f"👤 User: {user_input}")
    print("═" * 70)

    # ── Bước 1: Invoke lần đầu — agent plan + execute đến khi interrupt ──
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config,
        version="v2",
    )

    # ── Bước 2: Loop HIL ──
    while result.interrupts:
        interrupt_value = result.interrupts[0].value
        action_requests = interrupt_value["action_requests"]
        review_configs = {cfg["action_name"]: cfg for cfg in interrupt_value["review_configs"]}

        print(f"\n⛔ Cần duyệt {len(action_requests)} tool call(s):")
        decisions = []
        free_texts = []
        for i, action in enumerate(action_requests, 1):
            cfg = review_configs[action["name"]]
            print(f"\n  #{i} 🔧 {action['name']}")
            print(f"     Args: {json.dumps(action['args'], indent=6, ensure_ascii=False)}")
            decision, ft = get_human_decision(action, cfg)
            decisions.append(decision)
            if ft:
                free_texts.append(ft)

        print(f"\n📋 Decisions: {json.dumps(decisions, indent=2, ensure_ascii=False)}")

        # Resume — gửi decisions (tool bị reject)
        result = agent.invoke(
            Command(resume={"decisions": decisions}),
            config=config,
            version="v2",
        )

        # Nếu user có free-text, gửi tiếp — agent tự retry với thông tin mới
        for ft in free_texts:
            result = agent.invoke(
                {"messages": [{"role": "user", "content": ft}]},
                config=config,
                version="v2",
            )

    # ── Bước 3: In kết quả cuối ──
    print("\n" + "═" * 70)
    print(f"✅ Agent hoàn thành!")
    final_msg = result.value["messages"][-1]
    if hasattr(final_msg, "content") and final_msg.content:
        print(f"\n🤖 Agent: {final_msg.content}")


print("✅ Helper functions ready!")
print()
print("📌 `stream_agent()`     — xem plan + execution step-by-step (dùng stream_mode='values')")
print("📌 `run_plan_execute()` — full Plan-Execute + HIL loop")

✅ Helper functions ready!

📌 `stream_agent()`     — xem plan + execution step-by-step (dùng stream_mode='values')
📌 `run_plan_execute()` — full Plan-Execute + HIL loop


In [8]:
from langchain_core.messages.ai import AIMessage

In [29]:
# ── Hàm stream_with_hil: real-time + HIL ──

def _process_events(event_stream, seen_tool_calls):
    """Xử lý stream events và in real-time output.
    Trả về interrupts nếu có."""
    interrupts = None
    for event in event_stream:
        evt_interrupts = event.get("interrupts")
        if evt_interrupts:
            interrupts = evt_interrupts
        msgs = event.get("data", {}).get("messages", [])
        if not msgs:
            continue
        last = msgs[-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            for tc in last.tool_calls:
                tc_id = tc.get("id", "")
                if tc_id in seen_tool_calls:
                    continue
                seen_tool_calls.add(tc_id)
                if tc["name"] == "write_todos":
                    todos = tc["args"].get("todos", [])
                    print(f"\n📋 PLAN — {len(todos)} task(s):")
                    for i, t in enumerate(todos, 1):
                        label = t.get('title') or t.get('description') or t.get('content', '?')
                        print(f"   {i}. {label}")
                else:
                    args_str = json.dumps(tc["args"], ensure_ascii=False)
                    print(f"\n🔧 {tc['name']}({args_str})")
        if hasattr(last, "type") and last.type == "tool":
            print(f"   ✅ Result: {last.content[:]}")
        if isinstance(last, AIMessage) and last.content and not last.tool_calls:
            print(f"\n🤖 Agent: {last.content[:]}")
    return interrupts


def stream_with_hil(user_input: str, thread_id: str):
    """Real-time stream + HIL — vừa xem events real-time vừa hỏi user khi interrupt."""
    config = {"configurable": {"thread_id": thread_id}}
    print(f"👤 User: {user_input}")
    print("═" * 70)

    seen_tool_calls = set()
    first = True
    free_texts = []

    while True:
        # Lần đầu: gửi user input. Các lần sau: resume từ interrupt
        if first:
            stream_input = {"messages": [{"role": "user", "content": user_input}]}
            first = False
        else:
            stream_input = Command(resume={"decisions": decisions})

        # Stream events real-time
        events = agent.stream(
            stream_input, config=config, stream_mode="values", version="v2",
        )
        _process_events(events, seen_tool_calls)
        state = agent.get_state(config)

        # Nếu vừa resume và có free_texts, gửi tiếp → agent tự retry
        if free_texts:
            for ft in free_texts:
                events = agent.stream(
                    {"messages": [{"role": "user", "content": ft}]},
                    config=config, stream_mode="values", version="v2",
                )
                _process_events(events, seen_tool_calls)
            state = agent.get_state(config)
            free_texts = []

        if not state.interrupts:
            break

        # HIL: hỏi user duyệt
        iv = state.interrupts[0].value
        action_requests = iv["action_requests"]
        review_configs = {cfg["action_name"]: cfg for cfg in iv["review_configs"]}

        print(f"\n⛔ Cần duyệt {len(action_requests)} tool call(s):")
        decisions = []
        free_texts = []
        for i, action in enumerate(action_requests, 1):
            cfg = review_configs[action["name"]]
            print(f"\n  #{i} 🔧 {action['name']}")
            print(f"     Args: {json.dumps(action['args'], indent=6, ensure_ascii=False)}")
            decision, ft = get_human_decision(action, cfg)
            decisions.append(decision)
            if ft:
                free_texts.append(ft)

        print(f"\n📋 Decisions: {json.dumps(decisions, indent=2, ensure_ascii=False)}")

    print("\n" + "═" * 70)
    print(f"✅ Agent hoàn thành!")


print("✅ stream_with_hil() ready!")
print("📌 Vừa stream real-time, vừa hỗ trợ HIL interrupt")


✅ stream_with_hil() ready!
📌 Vừa stream real-time, vừa hỗ trợ HIL interrupt


### 5.0 stream_with_hil()

In [30]:
user_input = "Hãy lập kế hoạch và thực hiện chiến dịch email marketing cho khách hàng C001, C002, C003, giới thiệu sản phẩm P002 và P003. Tổng hợp kết quả cuối cùng."

thread_id = uuid7()

print(f"thread_id: {thread_id}")

thread_id: 019f12c0-d0da-7591-82f0-6a5ff3ff7edc


In [ ]:
stream_with_hil(user_input, thread_id)

👤 User: Hãy lập kế hoạch và thực hiện chiến dịch email marketing cho khách hàng C001, C002, C003, giới thiệu sản phẩm P002 và P003. Tổng hợp kết quả cuối cùng.
══════════════════════════════════════════════════════════════════════

📋 PLAN — 5 task(s):
   1. Tra cứu thông tin khách hàng C001, C002, C003
   2. Tra cứu thông tin sản phẩm P002, P003
   3. Soạn nội dung email chiến dịch marketing
   4. Gửi email chiến dịch cho từng khách hàng
   5. Tổng hợp kết quả và tạo báo cáo

🔧 search_customer({"customer_id": "C001"})

🔧 search_customer({"customer_id": "C002"})

🔧 search_customer({"customer_id": "C003"})

🔧 search_product({"product_id": "P002"})

🔧 search_product({"product_id": "P003"})
   ✅ Result: {
  "name": "AI API Access",
  "price": 5000000,
  "stock": 20
}

📋 PLAN — 5 task(s):
   1. Tra cứu thông tin khách hàng C001, C002, C003
   2. Tra cứu thông tin sản phẩm P002, P003
   3. Soạn nội dung email chiến dịch marketing
   4. Gửi email chiến dịch cho từng khách hàng
   5. Tổng hợp 

In [157]:
config = {"configurable": {"thread_id": thread_id}}
seen_tool_calls = set()
first = True

In [158]:
# Lần đầu: gửi user input. Các lần sau: resume từ interrupt
if first:
    stream_input = {"messages": [{"role": "user", "content": user_input}]}
    first = False
# else:
#     stream_input = Command(resume={"decisions": decisions})

In [159]:
events = agent.stream(
    stream_input, config=config, stream_mode="values", version="v2",
)
# _process_events(events, seen_tool_calls)

In [160]:
_process_events(events, seen_tool_calls)


📋 PLAN — 6 task(s):
   1. Tra cứu thông tin khách hàng C001, C002, C003
   2. Tra cứu thông tin sản phẩm P002, P003
   3. Soạn nội dung email marketing giới thiệu sản phẩm
   4. Xác nhận nội dung email với người dùng
   5. Gửi chiến dịch email marketing
   6. Tổng hợp kết quả và tạo báo cáo
   ✅ Result: Updated todo list to [{'content': 'Tra cứu thông tin khách hàng C001, C002, C003', 'status': 'in_progress'}, {'content': 'Tra cứu thông tin sản phẩm P002, P003', 'status': 'pending'}, {'content': 'Soạn nội dung email marketing giới thiệu sản phẩm', 'status': 'pending'}, {'content': 'Xác nhận nội dung email với người dùng', 'status': 'pending'}, {'content': 'Gửi chiến dịch email marketing', 'status': 'pending'}, {'content': 'Tổng hợp kết quả và tạo báo cáo', 'status': 'pending'}]

🔧 search_customer({"customer_id": "C001"})

🔧 search_customer({"customer_id": "C002"})

🔧 search_customer({"customer_id": "C003"})
   ✅ Result: {
  "name": "Le Van C",
  "email": "c@example.com",
  "plan": "En

In [166]:
state = agent.get_state(config)
state.values

{'messages': [HumanMessage(content='Hãy lập kế hoạch và thực hiện chiến dịch email marketing cho khách hàng C001, C002, C003, giới thiệu sản phẩm P002 và P003. Tổng hợp kết quả cuối cùng.', additional_kwargs={}, response_metadata={}, id='73af723d-0986-40b5-8bc4-a183cd448779'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 321, 'prompt_tokens': 6734, 'total_tokens': 7055, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'v_glm46', 'system_fingerprint': None, 'id': 'chatcmpl-87358c29c142fe5a', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f12a7-fc0e-7471-9d87-7ea0f0857751-0', tool_calls=[{'name': 'write_todos', 'args': {'todos': [{'content': 'Tra cứu thông tin khách hàng C001, C002, C003', 'status': 'in_progress'}, {'content': 'Tra cứu thông tin sản phẩm P002, P003', 'status': 'pending'}, {'content': 'Soạn nội dung email marketing giới thiệ

In [167]:
state.interrupts

()

In [171]:
events = agent.stream(
    Command(resume= "đồng ý"),
    config=config, stream_mode="values", version="v2"
)
_process_events(events, seen_tool_calls)


🤖 Agent: 

Tôi đã thu thập thông tin cần thiết. Dưới đây là tóm tắt:

**Thông tin khách hàng:**
- C001: Nguyen Van A (a@example.com) - Premium plan, đã chi 25 triệu
- C002: Tran Thi B (b@example.com) - Basic plan, đã chi 3.5 triệu
- C003: Le Van C (c@example.com) - Enterprise plan, đã chi 120 triệu

**Thông tin sản phẩm:**
- P002: Cloud Storage Pro - 2 triệu VNĐ, còn 50 sản phẩm
- P003: AI API Access - 5 triệu VNĐ, còn 20 sản phẩm

**Nội dung email dự kiến:**

```
Chào [Tên khách hàng],

Chúng tôi xin giới thiệu đến bạn hai sản phẩm mới:

1. Cloud Storage Pro (P002) - 2.000.000 VNĐ
   - Giải pháp lưu trữ đám mây chuyên nghiệp
   - Dung lượng lớn, bảo mật cao

2. AI API Access (P003) - 5.000.000 VNĐ
   - Truy cập API AI tiên tiến
   - Tích hợp dễ dàng, hiệu suất cao

Liên hệ ngay để nhận tư vấn miễn phí!

Trân trọng,
Đội ngũ Marketing
```

Bạn vui lòng xem xét và xác nhận nội dung email này trước khi tôi tiến hành gửi chiến dịch.


In [172]:
state.interrupts

()

### 5.1 Understand stream_agent

In [36]:
user_input = "Hãy lập kế hoạch và thực hiện chiến dịch email marketing cho khách hàng C001, C002, C003, giới thiệu sản phẩm P002 và P003. Tổng hợp kết quả cuối cùng."

thread_id = uuid7()

thread_id

UUID('019f1263-8d5b-7ad1-9e7c-c616b90caa99')

In [37]:
config = {"configurable": {"thread_id": thread_id}}
seen_tool_calls = set()

In [38]:

# def stream_agent(user_input: str, thread_id: str):
"""Stream agent execution — hiển thị plan, tool calls, và kết quả step-by-step.

Dùng stream_mode='values' để lấy toàn bộ state sau mỗi bước.
"""

seen_tool_calls = set()

for event in agent.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config,
        stream_mode="values",
        version="v2",
    ):
    
    # Messages nằm trong event['data']['messages'], không phải event['messages']
    msgs = event.get("data", {}).get("messages", [])
    if not msgs:
        continue
    last = msgs[-1]

    break

In [39]:
hasattr(last, "tool_calls") and last.tool_calls

False

In [40]:
from langchain_core.messages.ai import AIMessage

In [41]:

# def stream_agent(user_input: str, thread_id: str):
"""Stream agent execution — hiển thị plan, tool calls, và kết quả step-by-step.

Dùng stream_mode='values' để lấy toàn bộ state sau mỗi bước.
"""

seen_tool_calls = set()

for event in agent.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config,
        stream_mode="values",
        version="v2",
    ):
    
    # Messages nằm trong event['data']['messages'], không phải event['messages']
    msgs = event.get("data", {}).get("messages", [])
    if not msgs:
        continue
    last = msgs[-1]

    # Tool call — highlight todos vs regular tools
    if hasattr(last, "tool_calls") and last.tool_calls:
        for tc in last.tool_calls:
            # Dedup bằng id
            tc_id = tc.get("id", "")
            if tc_id in seen_tool_calls:
                continue
            seen_tool_calls.add(tc_id)

            if tc["name"] == "write_todos":
                todos = tc["args"].get("todos", [])

                print(f"\nPLAN — {len(todos)} task(s):")
                for i, t in enumerate(todos, 1):
                    # write_todos có thể dùng content, title, hoặc description
                    label = t.get('title') or t.get('description') or t.get('content', '?')
                    print(f"   {i}. {label}")
            else:
                args_str = json.dumps(tc["args"], ensure_ascii=False)
                print(f"\n🔧 {tc['name']}({args_str})")

    # Tool result
    if hasattr(last, "type") and last.type == "tool":
        print(f"   ✅ Result: {last.content[:]}")

    # Agent trả lời text (chỉ khi không phải republish)
    if isinstance(last, AIMessage) and last.content and not last.tool_calls:
        print(f"\n🤖 Agent: {last.content[:]}")

# print("\n" + "═" * 70)


PLAN — 5 task(s):
   1. Tra cứu thông tin khách hàng C001, C002, C003
   2. Tra cứu thông tin sản phẩm P002, P003
   3. Soạn nội dung email chiến dịch marketing
   4. Gửi email chiến dịch cho từng khách hàng
   5. Tổng hợp kết quả chiến dịch
   ✅ Result: Updated todo list to [{'content': 'Tra cứu thông tin khách hàng C001, C002, C003', 'status': 'in_progress'}, {'content': 'Tra cứu thông tin sản phẩm P002, P003', 'status': 'pending'}, {'content': 'Soạn nội dung email chiến dịch marketing', 'status': 'pending'}, {'content': 'Gửi email chiến dịch cho từng khách hàng', 'status': 'pending'}, {'content': 'Tổng hợp kết quả chiến dịch', 'status': 'pending'}]

🔧 search_customer({"customer_id": "C001"})

🔧 search_customer({"customer_id": "C002"})

🔧 search_customer({"customer_id": "C003"})
   ✅ Result: {
  "name": "Le Van C",
  "email": "c@example.com",
  "plan": "Enterprise",
  "spent_2026": 120000000
}

PLAN — 5 task(s):
   1. Tra cứu thông tin khách hàng C001, C002, C003
   2. Tra cứu thông

### 5.2 - Understand run_plan_execute

In [67]:
user_input = "Hãy lập kế hoạch và thực hiện chiến dịch email marketing cho khách hàng C001, C002, C003, giới thiệu sản phẩm P002 và P003. gửi email cho khách hàng C001, C002, C003. Tổng hợp kết quả cuối cùng."

thread_id = uuid7()

In [68]:
"""Full Plan-Execute + HIL loop."""
config = {"configurable": {"thread_id": thread_id}}

In [83]:
# ── Bước 1: Invoke lần đầu — agent plan + execute đến khi interrupt ──
result = agent.invoke(
    {"messages": [{"role": "user", "content": user_input}]},
    config=config,
    version="v2",
)

In [70]:
result.value.keys()

dict_keys(['messages', 'todos', 'files'])

In [71]:
result.value['todos']

[{'content': 'Tra cứu thông tin khách hàng C001, C002, C003',
  'status': 'completed'},
 {'content': 'Tra cứu thông tin sản phẩm P002, P003', 'status': 'completed'},
 {'content': 'Soạn nội dung email marketing giới thiệu sản phẩm',
  'status': 'in_progress'},
 {'content': 'Gửi email chiến dịch cho khách hàng (chờ phê duyệt)',
  'status': 'pending'},
 {'content': 'Tổng hợp kết quả và tạo báo cáo', 'status': 'pending'}]

In [72]:
result.value['files']

{}

In [84]:
for msg in result.value['messages']:
    msg.pretty_print()

================================ Human Message =================================

Hãy lập kế hoạch và thực hiện chiến dịch email marketing cho khách hàng C001, C002, C003, giới thiệu sản phẩm P002 và P003. gửi email cho khách hàng C001, C002, C003. Tổng hợp kết quả cuối cùng.
================================== Ai Message ==================================
Tool Calls:
  write_todos (chatcmpl-tool-968b833cd312a28a)
 Call ID: chatcmpl-tool-968b833cd312a28a
  Args:
    todos: [{'content': 'Tra cứu thông tin khách hàng C001, C002, C003', 'status': 'in_progress'}, {'content': 'Tra cứu thông tin sản phẩm P002, P003', 'status': 'pending'}, {'content': 'Soạn nội dung email marketing giới thiệu sản phẩm', 'status': 'pending'}, {'content': 'Gửi email chiến dịch cho khách hàng (chờ phê duyệt)', 'status': 'pending'}, {'content': 'Tổng hợp kết quả và tạo báo cáo', 'status': 'pending'}]
================================= Tool Message =================================
Name: write_todos

Updated todo li

In [90]:
result.interrupts[0].value

{'action_requests': [{'name': 'send_email_campaign',
   'args': {'to': 'a@example.com',
    'subject': 'Giới thiệu sản phẩm mới: Cloud Storage Pro & AI API Access',
    'body': 'Kính gửi Quý khách,\n\nChúng tôi xin giới thiệu đến Quý khách hai sản phẩm mới nhất:\n\n1. Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp\n   - Giá: 2.000.000 VND\n   - Ưu điểm: Dung lượng lớn, bảo mật cao, truy cập mọi lúc mọi nơi\n\n2. AI API Access - Truy cập API trí tuệ nhân tạo\n   - Giá: 5.000.000 VND\n   - Ưu điểm: Tích hợp AI mạnh mẽ, hỗ trợ đa nền tảng, API linh hoạt\n\nCả hai sản phẩm đều đang có sẵn hàng và sẵn sàng phục vụ Quý khách.\n\nNếu có bất kỳ câu hỏi nào, vui lòng liên hệ với chúng tôi.\n\nTrân trọng,\nĐội ngũ Hỗ trợ Khách hàng'},
   'description': "Tool execution requires approval\n\nTool: send_email_campaign\nArgs: {'to': 'a@example.com', 'subject': 'Giới thiệu sản phẩm mới: Cloud Storage Pro & AI API Access', 'body': 'Kính gửi Quý khách,\\n\\nChúng tôi xin giới thiệu đến Quý 

In [ ]:

# def run_plan_execute(user_input: str, thread_id: str):
"""Full Plan-Execute + HIL loop."""
config = {"configurable": {"thread_id": thread_id}}

print(f"👤 User: {user_input}")
print("═" * 70)

# ── Bước 1: Invoke lần đầu — agent plan + execute đến khi interrupt ──
result = agent.invoke(
    {"messages": [{"role": "user", "content": user_input}]},
    config=config,
    version="v2",
)

# ── Bước 2: Loop HIL ──
while result.interrupts:
    interrupt_value = result.interrupts[0].value
    action_requests = interrupt_value["action_requests"]
    review_configs = {cfg["action_name"]: cfg for cfg in interrupt_value["review_configs"]}

    print(f"\n⛔ Cần duyệt {len(action_requests)} tool call(s):")
    decisions = []
    for i, action in enumerate(action_requests, 1):
        cfg = review_configs[action["name"]]
        print(f"\n  #{i} 🔧 {action['name']}")
        print(f"     Args: {json.dumps(action['args'], indent=6, ensure_ascii=False)}")
        decision = get_human_decision(action, cfg)
        decisions.append(decision)

    print(f"\n📋 Decisions: {json.dumps(decisions, indent=2, ensure_ascii=False)}")

    # Resume
    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,
        version="v2",
    )

# ── Bước 3: In kết quả cuối ──
print("\n" + "═" * 70)
print(f"✅ Agent hoàn thành!")
final_msg = result.value["messages"][-1]
if hasattr(final_msg, "content") and final_msg.content:
    print(f"\n🤖 Agent: {final_msg.content}")



### 5.3 stream_with_hil() — Real-time Stream + HIL

Khác với `stream_agent()` (chỉ stream, không HIL) và `run_plan_execute()` (invoke, có HIL, không real-time):

`stream_with_hil()` kết hợp cả hai:
- ✅ **Real-time**: thấy 📋 PLAN, 🔧 tool calls, ✅ results ngay lập tức
- ✅ **HIL**: khi gặp tool dangerous → pause, hỏi user → resume → stream tiếp

> 🔄 Dùng `agent.stream()` + `agent.get_state()` + `Command(resume=...)` trong một loop duy nhất.

In [ ]:
# Demo stream_with_hil — real-time events + HIL
stream_with_hil(
    user_input="""Look up customer C001, check product P002.
Draft and send an email campaign to a@example.com:
  Subject: 'Premium offer'
  Body: 'Dear customer, check our premium offer!'
Then generate a summary report.""",
    thread_id="stream-hil-demo-1",
)

👤 User: Look up customer C001, check product P002.
Draft and send an email campaign to a@example.com:
  Subject: 'Premium offer'
  Body: 'Dear customer, check our premium offer!'
Then generate a summary report.
══════════════════════════════════════════════════════════════════════

🔧 send_email_campaign({"to": "a@example.com", "subject": "Premium offer", "body": "Dear customer, check our premium offer!"})

⛔ Cần duyệt 1 tool call(s):

  #1 🔧 send_email_campaign
     Args: {
      "to": "a@example.com",
      "subject": "Premium offer",
      "body": "Dear customer, check our premium offer!"
}
  Options: approve, edit, reject

📋 Decisions: [
  {
    "type": "approve"
  }
]
  📧 Sending campaign email → a@example.com
   ✅ Result: ✅ Campaign email sent to a@example.com: 'Premium offer'

📋 PLAN — 4 task(s):
   1. Tra cứu thông tin khách hàng C001
   2. Tra cứu thông tin sản phẩm P002
   3. Gửi email chiến dịch đến a@example.com
   4. Tạo báo cáo tổng kết

🔧 generate_report({"report_type": "

## 6. Demo 1 — Stream: Xem Agent Plan & Execute Step-by-Step

Dùng `stream_agent()` để thấy agent **plan** trước (gọi `write_todos`), rồi execute từng bước.  
Mode này **stream real-time** nhưng không có HIL — dùng để quan sát hành vi.

> 💡 Chạy cell này trước để xem agent tự động plan như thế nào.

In [15]:
# Dùng stream_agent để xem agent plan & execute step-by-step (không HIL)
# Dùng thread_id mới để tránh conflict với lần chạy trước
stream_agent(
    user_input="Search customer C001, check what products we have, "
               "then generate a sales report for them.",
    thread_id="plan-exec-stream-5",
)

👤 User: Search customer C001, check what products we have, then generate a sales report for them.
══════════════════════════════════════════════════════════════════════

📋 PLAN — 3 task(s):
   1. Tra cứu thông tin khách hàng C001
   2. Kiểm tra danh sách sản phẩm có sẵn
   3. Tạo báo cáo doanh số cho khách hàng C001

🔧 search_customer({"customer_id": "C001"})
   ✅ Result: {
  "name": "Nguyen Van A",
  "email": "a@example.com",
  "plan": "Premium",
  "spent_2026": 25000000
}

📋 PLAN — 3 task(s):
   1. Tra cứu thông tin khách hàng C001
   2. Kiểm tra danh sách sản phẩm có sẵn
   3. Tạo báo cáo doanh số cho khách hàng C001

🔧 generate_report({"report_type": "sales_report", "data_summary": "Customer C001 (Nguyen Van A) - Premium Plan - Total spent in 2026: 25,000,000 VND. Available products: Cloud Storage Basic (500,000 VND), Cloud Storage Pro (2,000,000 VND), AI API Access (5,000,000 VND)."})
   ✅ Result: 📄 Report 'sales_report' generated. Summary: Customer C001 (Nguyen Van A) - Premium P

## 7. Demo 2 — Plan-Execute + HIL: Yêu Cầu Phức Hợp

Flow đầy đủ:

```
1️⃣ User: "Tra cứu C001, xem sản phẩm, gửi email upsell, báo cáo"
2️⃣ Agent: write_todos → [Tra cứu, Xem SP, Soạn email, Gửi email, Báo cáo]
3️⃣ Execute: Tra cứu ✅ → Xem SP ✅ → Soạn email ✅ → ⛔ Gửi email (HIL)
4️⃣ User:      duyệt nội dung email → Approve / Edit / Reject
5️⃣ Agent:     tiếp tục → Báo cáo ✅
```

Chạy cell dưới, nhập `a` (approve), `e` (edit), hoặc `r` (reject) khi được hỏi.

In [20]:
# Plan-Execute + HIL — yêu cầu phức hợp
run_plan_execute(
    user_input="""Look up customer C001 (Nguyen Van A) who is on Premium plan.
Search for product P002 (Cloud Storage Pro).
Draft and send an email campaign to a@example.com:
  Subject: 'Upgrade your storage with Cloud Pro'
  Body: 'Dear Nguyen Van A, as a Premium customer you get 20%% off Cloud Storage Pro!'
Finally, generate a report summarizing what was done.""",
    thread_id="plan-exec-hil-1",
)

👤 User: Look up customer C001 (Nguyen Van A) who is on Premium plan.
Search for product P002 (Cloud Storage Pro).
Draft and send an email campaign to a@example.com:
  Subject: 'Upgrade your storage with Cloud Pro'
  Body: 'Dear Nguyen Van A, as a Premium customer you get 20%% off Cloud Storage Pro!'
Finally, generate a report summarizing what was done.
══════════════════════════════════════════════════════════════════════

⛔ Cần duyệt 1 tool call(s):

  #1 🔧 send_email_campaign
     Args: {
      "to": "a@example.com",
      "subject": "Upgrade your storage with Cloud Pro",
      "body": "Dear Nguyen Van A, as a Premium customer you get 20% off Cloud Storage Pro!"
}
  Options: approve, edit, reject

📋 Decisions: [
  {
    "type": "edit",
    "edited_action": {
      "name": "send_email_campaign",
      "args": {
        "to": "tamtt@vnpay.v",
        "subject": "hkhj",
        "body": "kasdhflkhasdf"
      }
    }
  }
]
  📧 Sending campaign email → tamtt@vnpay.v

⛔ Cần duyệt 1 tool cal

## 8. Demo 3 — Batch Interrupt: Nhiều Email Cùng Lúc

Agent cần gửi email cho nhiều khách hàng — mỗi email là một tool call dangerous.
Tất cả được gom vào một interrupt batch. Bạn duyệt từng cái một.

In [ ]:
# Batch: nhiều email campaign trong một lần
run_plan_execute(
    user_input="""Send a promo email campaign to all our customers:
- a@example.com: subject 'Premium Flash Sale 30%' body 'Exclusive premium offer'
- b@example.com: subject 'Basic Upgrade Deal' body 'Upgrade to Premium now'
- c@example.com: subject 'Enterprise Special' body 'Volume pricing available'
Then generate a campaign summary report.""",
    thread_id="plan-exec-hil-batch",
)

## 9. Tổng Kết: Cơ Chế Plan-Execute của Deep Agents

### Luồng hoạt động

```
User Input
   │
   ▼
┌──────────────────────────────────────────┐
│  Agent gọi write_todos                   │
│  → Phân tích request → List các task     │
│  → Agent tự động execute từng task       │
└──────────────────────────────────────────┘
   │
   ▼  (cho mỗi task)
┌──────────────────────────────────────────┐
│  Agent gọi tool cần thiết                │
│  ├── Safe tool → chạy luôn              │
│  └── Dangerous tool                      │
│       → ⛔ INTERRUPT → Human duyệt      │
│       → Resume → Agent tiếp tục          │
└──────────────────────────────────────────┘
   │
   ▼
Agent tổng hợp kết quả → Final response
```

### Code pattern tối thiểu

```python
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver

agent = create_deep_agent(
    model=...,
    tools=[...],
    checkpointer=MemorySaver(),
    interrupt_on={"dangerous_tool": {"allowed_decisions": ["approve", "edit", "reject"]}},
    system_prompt="Always plan with write_todos first, then execute step by step.",
)

# Invoke
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Complex request..."}]},
    config={"configurable": {"thread_id": "my-thread"}},
    version="v2",
)

# HIL loop
while result.interrupts:
    decisions = [...]  # user approve/edit/reject
    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,
        version="v2",
    )
```

### Lưu ý quan trọng

| Khía cạnh | Giải thích |
|-----------|-----------|
| **`write_todos` là built-in** | Agent tự động gọi — không cần định nghĩa |
| **Plan tự nhiên** | Agent tự quyết định khi nào plan, không cần chỉ thị cứng |
| **HIL trong execution** | `interrupt_on` chỉ ảnh hưởng đến lúc execute, không ảnh hưởng đến planning |
| **Cùng thread_id** | Bắt buộc để resume đúng state |
| **Decisions match order** | Mảng decisions phải cùng thứ tự action_requests |